In [1]:
!pip install uvicorn 


[notice] A new release of pip is available: 23.2.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [2]:

import json
import os
import torch
import umap
import numpy as np
from fastapi import FastAPI
from sklearn.cluster import KMeans
from starlette.middleware.cors import CORSMiddleware
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
import time
import tracemalloc

/Users/maneet/Documents/MAI/3D-visualization of High Dimentional Data/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
root_folder_path = '../Test_data'

In [4]:
data_list = []
file_names = []
labels = []

## Traverse Directory Tree for .pt Files

The following Python code snippet demonstrates how to recursively traverse a directory tree (`root_folder_path`) and process `.pt` files using the `os.walk` function from the `os` module and PyTorch (`torch`). It loads tensor data from each `.pt` file found, flattens it to a NumPy array, and collects the data along with associated file names and labels.


In [5]:
# Traverse the directory tree
for root, dirs, files in os.walk(root_folder_path):
    for file in files:
        if file.endswith('.pt'):
            file_path = os.path.join(root, file)
            try:
                # Load the tensor onto the CPU
                data = torch.load(file_path, map_location=torch.device('cpu'))
                if isinstance(data, torch.Tensor):
                    data_list.append(data.numpy().flatten())
                    file_names.append(os.path.basename(file_path))
                    labels.append(os.path.basename(root))
            except Exception as e:
                print(f"Error loading {file_path}: {e}")

# Check if any data was loaded
if not data_list:
    raise ValueError("No .pt files found in the specified directory and its subdirectories.")

## Calculate Maximum Length of Data in `data_list`

The following Python code snippet calculates the maximum length of data stored in `data_list`.


In [6]:
# Determine the maximum length of the flattened tensors
max_length = max(len(data) for data in data_list)

## Pad Data in `data_list` to Match `max_length`

The following Python code snippet pads each element in `data_list` to match `max_length` using NumPy's `np.pad` function.


In [7]:
# Pad all tensors to the maximum length
padded_data_list = [np.pad(data, (0, max_length - len(data)), 'constant') for data in data_list]

## Stack Padded Data from `padded_data_list` into `all_data`

The following Python code snippet stacks the padded data from `padded_data_list` into a single NumPy array `all_data` using `np.vstack`.


In [8]:
# Concatenate all data into a single numpy array
all_data = np.vstack(padded_data_list)

## Function to Create JSON Data from Embeddings

The following Python function `create_json_data` converts embeddings along with associated metadata into a structured JSON format.


In [9]:
# Function to create JSON data from embeddings
def create_json_data(embedding, file_names, labels, memory, silhouette, time):
    json_data = {
        "data": [],
        "performance": {
            "memory": float(memory),
            "silhouette": float(silhouette),
            "time": float(time)
        }
    }
    for i in range(len(embedding)):
        json_entry = {
            'x': float(embedding[i, 0]),
            'y': float(embedding[i, 1]),
            'z': float(embedding[i, 2]),
            'filename': file_names[i],
            'label': str(labels[i])  # Ensure label is a string
        }
        json_data["data"].append(json_entry)
    return json.dumps(json_data, indent=4)

## Measure Time and Memory for Dimensionality Reduction Techniques

The following Python code snippet measures the time taken and memory usage for different dimensionality reduction techniques (`UMAP`, `t-SNE`, and `PCA`) applied to `all_data`.


In [10]:
# Measure time and memory for UMAP
tracemalloc.start()
start_time = time.time()
umap_embedding = umap.UMAP(n_components=3).fit_transform(all_data)
umap_time = time.time() - start_time
umap_current, umap_peak = tracemalloc.get_traced_memory()
tracemalloc.stop()
umap_memory = umap_peak / 1024 / 1024  # Convert to MiB

# Measure time and memory for t-SNE
tracemalloc.start()
start_time = time.time()
tsne_embedding = TSNE(n_components=3).fit_transform(all_data)
tsne_time = time.time() - start_time
tsne_current, tsne_peak = tracemalloc.get_traced_memory()
tracemalloc.stop()
tsne_memory = tsne_peak / 1024 / 1024  # Convert to MiB

# Measure time and memory for PCA
tracemalloc.start()
start_time = time.time()
pca_embedding = PCA(n_components=3).fit_transform(all_data)
pca_time = time.time() - start_time
pca_current, pca_peak = tracemalloc.get_traced_memory()
tracemalloc.stop()
pca_memory = pca_peak / 1024 / 1024  # Convert to MiB


In [11]:
print(f'Shape of concatnated input data to the dimentionality reduction process: {all_data.shape}')
print(f'Shape of output data from dimentionality reduction process: {umap_embedding.shape}')
print(f'Output data after dimentionality reduction: {umap_embedding}')


Shape of concatnated input data to the dimentionality reduction process: (3266, 65536)
Shape of output data from dimentionality reduction process: (3266, 3)
Output data after dimentionality reduction: [[15.798419   5.7126303  7.392352 ]
 [16.824669   5.873009   4.5886245]
 [17.2073     6.1646967  5.3314176]
 ...
 [-1.5290879  2.1014345  4.624107 ]
 [-2.0341115  1.7765654  3.2308586]
 [-1.398462   1.3484461  3.827828 ]]


## Cluster Data Using KMeans and Calculate Silhouette Scores

The following Python code snippet clusters data using KMeans and calculates silhouette scores for different embeddings (`UMAP`, `t-SNE`, `PCA`).


In [12]:
# Cluster the data using KMeans for silhouette score calculation
kmeans = KMeans(n_clusters=3, random_state=42)
cluster_labels = kmeans.fit_predict(umap_embedding)

umap_silhouette = silhouette_score(umap_embedding, cluster_labels)
tsne_silhouette = silhouette_score(tsne_embedding, cluster_labels)
pca_silhouette = silhouette_score(pca_embedding, cluster_labels)

## Create JSON Data from Embeddings

The following Python code snippet creates JSON data from embeddings (`UMAP`, `t-SNE`, `PCA`) along with associated metadata.


In [13]:
# Create JSON data
umap_json_data = create_json_data(umap_embedding, file_names, labels, umap_memory, umap_silhouette, umap_time)
tsne_json_data = create_json_data(tsne_embedding, file_names, labels, tsne_memory, tsne_silhouette, tsne_time)
pca_json_data = create_json_data(pca_embedding, file_names, labels, pca_memory, pca_silhouette, pca_time)


In [14]:

import uvicorn

## Install and Apply nest_asyncio for Jupyter Notebooks

In a Jupyter notebook or IPython environment, you can handle multiple event loops using the nest_asyncio package. This package allows for asynchronous tasks to be run in environments that already have an event loop running.

In [15]:
!pip install nest_asyncio

import nest_asyncio
nest_asyncio.apply()


[notice] A new release of pip is available: 23.2.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


## Setting Up a FastAPI Application with CORS

The following Python code sets up a FastAPI application (`app`) with CORS (Cross-Origin Resource Sharing) enabled to handle requests for `UMAP`, `t-SNE`, and `PCA` JSON data endpoints.

**Note:** Modify the `origins` list to match the ports and domains you intend to allow CORS requests from.


In [ ]:
# FastAPI app setup
app = FastAPI()

# CORS setup
origins = [
    "http://127.0.0.1",
    "http://127.0.0.1:5500",
]

# CORS middleware to allow cross-origin requests
app.add_middleware(
    CORSMiddleware,
    allow_origins=origins,
    allow_credentials=True,
    allow_methods=["GET", "POST", "PUT", "DELETE"],
    allow_headers=["*"],
)

@app.get("/my-umap")
def umap_endpoint():
    return json.loads(umap_json_data)

@app.get("/my-tsne")
def tsne_endpoint():
    return json.loads(tsne_json_data)

@app.get("/my-pca")
def pca_endpoint():
    return json.loads(pca_json_data)

# Start the server in notebooks without asyncio.run()
import uvicorn
server = uvicorn.Server(
    uvicorn.Config(app, host="127.0.0.1", port=8000, log_level="info")
)
await server.serve()

INFO:     Started server process [76934]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:60051 - "GET /my-pca HTTP/1.1" 200 OK
INFO:     127.0.0.1:60084 - "GET /my-tsne HTTP/1.1" 200 OK
INFO:     127.0.0.1:60088 - "GET /my-umap HTTP/1.1" 200 OK
INFO:     127.0.0.1:60099 - "GET /my-pca HTTP/1.1" 200 OK
INFO:     127.0.0.1:60101 - "GET /my-tsne HTTP/1.1" 200 OK
INFO:     127.0.0.1:60103 - "GET /my-umap HTTP/1.1" 200 OK
INFO:     127.0.0.1:60142 - "GET /my-pca HTTP/1.1" 200 OK
INFO:     127.0.0.1:60167 - "GET /my-pca HTTP/1.1" 200 OK
INFO:     127.0.0.1:60171 - "GET /my-pca HTTP/1.1" 200 OK
INFO:     127.0.0.1:60188 - "GET /my-pca HTTP/1.1" 200 OK
INFO:     127.0.0.1:60188 - "GET /my-pca HTTP/1.1" 200 OK
INFO:     127.0.0.1:60198 - "GET /my-pca HTTP/1.1" 200 OK
INFO:     127.0.0.1:60206 - "GET /my-pca HTTP/1.1" 200 OK
INFO:     127.0.0.1:60206 - "GET /my-pca HTTP/1.1" 200 OK
INFO:     127.0.0.1:60206 - "GET /my-pca HTTP/1.1" 200 OK
